In [4]:
import os
import re
import json
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer
from scipy.stats import wilcoxon
from tqdm import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device in uso: {device}\n")

BASE_INPUT_PATH = "/kaggle/input/datasets/claudiacornacchia/"
EMBEDDING_DIR = "embedding"
csv_path = os.path.join(BASE_INPUT_PATH, EMBEDDING_DIR, "base_map.csv")
json_path = os.path.join(BASE_INPUT_PATH, EMBEDDING_DIR, "bertopic_data.json")

emb_model_id = "Qwen/Qwen3-Embedding-0.6B"
print(f"Caricamento del modello {emb_model_id}...")
emb_tokenizer = AutoTokenizer.from_pretrained(emb_model_id, trust_remote_code=True)
emb_model = AutoModel.from_pretrained(emb_model_id, trust_remote_code=True).to(device)
emb_model.eval()

def get_embedding(text):
    inputs = emb_tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)
    with torch.no_grad():
        outputs = emb_model(**inputs)
        attention_mask = inputs['attention_mask']
        token_embeddings = outputs.last_hidden_state
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
        sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
        embedding = sum_embeddings / sum_mask
        return F.normalize(embedding, p=2, dim=1)

def load_bertopic_keywords(json_file):
    with open(json_file, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    topic_keywords = {}
    for topic in data:
        t_id = int(topic['id'])
        keywords = [item['word'] for item in topic.get('words_no_brands', topic.get('words_raw', []))]
        topic_keywords[t_id] = keywords
    return topic_keywords

def get_custom_stopwords():
    airline_stopwords = ["united", "usairways", "americanair", "southwestair", "jetblue", "virginamerica", "virgin", "southwest", "american", "delta", "us", "airways", "airlines"]
    polite_stopwords = ["dm", "please", "thanks", "thank", "sent", "send", "follow", "following", "amp", "http", "https", "co", "flight", "just"]
    return list(ENGLISH_STOP_WORDS) + airline_stopwords + polite_stopwords

def remove_targeted_brand(text, airline):
    mapping = {
        'united': 'united', 'delta': 'jetblue', 'jetblue': 'jetblue', 
        'us airways': 'usairways', 'american': 'americanair', 
        'southwest': 'southwestair', 'virgin america': 'virginamerica'
    }
    target = mapping.get(str(airline).lower(), str(airline).lower())
    return re.sub(rf'@?{re.escape(target)}\b', '', str(text), flags=re.IGNORECASE).strip()

def run_hybrid_tweet_ablation(df, topic_keywords_dict):
    results = []
    custom_stopwords = get_custom_stopwords()
    
    df_valid = df[df['airline'].notna() & df['text'].notna() & df['topic_id'].notna() & (df['topic_id'] != -1)].copy()
    
    grouped = df_valid.groupby(['airline', 'topic_id'])
    fallback_cluster_keywords = {}
    
    print("Pre-calcolo dei fallback TF-IDF per ogni sub-cluster...")
    for (airline, topic_id), group in grouped:
        texts = group['text'].astype(str).tolist()
        try:
            vectorizer = TfidfVectorizer(stop_words=custom_stopwords, max_features=3, ngram_range=(1,2))
            X = vectorizer.fit_transform(texts)
            scores = np.asarray(X.sum(axis=0)).ravel()
            sorted_indices = scores.argsort()[::-1]
            feature_names = vectorizer.get_feature_names_out()
            fallback_cluster_keywords[(airline, topic_id)] = [feature_names[idx] for idx in sorted_indices]
        except ValueError:
            fallback_cluster_keywords[(airline, topic_id)] = []

    print("\n Avvio dell'Ablation Study sui singoli Tweet...")
    for idx, row in tqdm(df_valid.iterrows(), total=len(df_valid), desc="Processing Tweets"):
        text = str(row['text'])
        airline = row['airline']
        topic = int(row['topic_id'])
        
        matched_kw = None
        source_method = "BERTopic_JSON"
        
        json_kws = topic_keywords_dict.get(topic, [])
        for kw in json_kws:
            if re.search(rf'\b{re.escape(kw)}\b', text, re.IGNORECASE):
                matched_kw = kw
                break
                
        if not matched_kw:
            source_method = "TF-IDF_Fallback"
            sub_cluster_kws = fallback_cluster_keywords.get((airline, topic), [])
            for kw in sub_cluster_kws:
                if re.search(rf'\b{re.escape(kw)}\b', text, re.IGNORECASE):
                    matched_kw = kw
                    break
                    
        if not matched_kw:
            continue
            
        no_brand = remove_targeted_brand(text, airline)
        no_op = re.sub(rf'\b{re.escape(matched_kw)}\b', '', text, flags=re.IGNORECASE).strip()
        
        if text != no_brand and text != no_op and len(no_brand) > 5 and len(no_op) > 5:
            orig_emb = get_embedding(text)
            emb_brand = get_embedding(no_brand)
            emb_op = get_embedding(no_op)
            
            results.append({
                'airline': airline,
                'topic': topic,
                'keyword': matched_kw,
                'method': source_method,
                'imp_brand': 1.0 - F.cosine_similarity(orig_emb, emb_brand).item(),
                'imp_op': 1.0 - F.cosine_similarity(orig_emb, emb_op).item()
            })
            
    df_res = pd.DataFrame(results)
    if df_res.empty:
        print("Nessun tweet idoneo per l'ablation.")
        return None
        
    stats = []
    for (airline, topic, kw), group in df_res.groupby(['airline', 'topic', 'keyword']):
        if len(group) >= 10:  
            imp_b = group['imp_brand'].values
            imp_o = group['imp_op'].values
            
            try:
                _, p_val = wilcoxon(imp_b, imp_o, zero_method='wilcox')
            except ValueError:
                p_val = 1.0
                
            effect_size = imp_b.mean() - imp_o.mean()
            coverage_pct = len(group) / len(df_valid[(df_valid['airline'] == airline) & (df_valid['topic_id'] == topic)])
            
            if p_val < 0.05:
                vincitore = 'Brand (Sig.)' if effect_size > 0 else 'Keyword (Sig.)'
            else:
                vincitore = 'Pareggio (Non Sig.)'
                
            # Contiamo quanti provengono da JSON e quanti da TF-IDF
            from_json = (group['method'] == 'BERTopic_JSON').sum()
            
            stats.append({
                'Airline': airline, 'Topic_ID': topic, 'Keyword': kw,
                'Tweet_Appaiati': len(group), 'From_JSON': from_json,
                'Mean_Impact_Brand': round(imp_b.mean(), 4), 'Mean_Impact_Key': round(imp_o.mean(), 4),
                'Effect_Size': round(effect_size, 4), 'P_Value': round(p_val, 5),
                'SubCluster_Coverage': round(coverage_pct, 2), 'Vincitore': vincitore
            })
            
    return pd.DataFrame(stats)

if __name__ == "__main__":
    try:
        df_dati = pd.read_csv(csv_path)
        print("File JSON BERTopic in caricamento...")
        topic_keywords_dict = load_bertopic_keywords(json_path)
        print("Dizionario delle keyword caricato con successo.")
        
        df_final_results = run_hybrid_tweet_ablation(df_dati, topic_keywords_dict)
        
        if df_final_results is not None:
            print("\n\n=== RISULTATI CON RIGORE STATISTICO (IBRIDO JSON + TF-IDF) ===")
            df_final_results = df_final_results.sort_values(by='P_Value')
            cols_to_print = ['Airline', 'Topic_ID', 'Keyword', 'Tweet_Appaiati', 'Mean_Impact_Brand', 'Mean_Impact_Key', 'P_Value', 'Vincitore']
            print(df_final_results[cols_to_print].to_string(index=False))
            
            vittorie_brand = (df_final_results['Vincitore'] == 'Brand (Sig.)').sum()
            vittorie_key = (df_final_results['Vincitore'] == 'Keyword (Sig.)').sum()
            pareggi = (df_final_results['Vincitore'] == 'Pareggio (Non Sig.)').sum()
            
            print(f"\n--- RIEPILOGO FINALE SU {len(df_final_results)} CONFRONTI VALIDI ---")
            print(f"Vittorie Brand (Statisticamente Significative): {vittorie_brand}")
            print(f"Vittorie Keyword (Statisticamente Significative): {vittorie_key}")
            print(f"Pareggi (Nessuna differenza statistica): {pareggi}")
            
            df_final_results.to_csv("hybrid_tweet_centric_ablation.csv", index=False)
            
    except FileNotFoundError as e:
        print(f"ERRORE di caricamento file: {e}")



Device in uso: cuda

Caricamento del modello Qwen/Qwen3-Embedding-0.6B...


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.19G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

File JSON BERTopic in caricamento...
Dizionario delle keyword caricato con successo.
Pre-calcolo dei fallback TF-IDF per ogni sub-cluster...

Avvio dell'Ablation Study sui singoli Tweet...


Processing Tweets: 100%|██████████| 5866/5866 [08:56<00:00, 10.93it/s]




=== RISULTATI CON RIGORE STATISTICO (IBRIDO JSON + TF-IDF) ===
       Airline  Topic_ID            Keyword  Tweet_Appaiati  Mean_Impact_Brand  Mean_Impact_Key  P_Value           Vincitore
      American         0             flight             163             0.0374           0.0183  0.00000        Brand (Sig.)
      American         0             thanks              41             0.1828           0.0286  0.00000        Brand (Sig.)
      American         2               hold              54             0.0638           0.0261  0.00000        Brand (Sig.)
      American         2              hours              23             0.0674           0.0085  0.00000        Brand (Sig.)
      American         4                bag              37             0.0597           0.0313  0.00000        Brand (Sig.)
      American         2              phone              42             0.0630           0.0258  0.00000        Brand (Sig.)
         Delta         0            delayed              19 